# CSV uzerinden sohbet

Araba tablosunu chroma'ya attim, fiyata gore soru sordum.


In [ ]:
import pandas as pd
import chromadb


### Data Dosyasını Okuma


In [ ]:
df=pd.read_excel('data/cars.xls')
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


### Boş satır


In [ ]:
df=df.dropna()


### Veritabani


In [ ]:
col=chromadb.PersistentClient(path='./cars_db').get_or_create_collection('car_data')

# her satiri aranabilir metin yap
docs=df.apply(lambda r: f'{r.Price},{r.Mileage},{r.Make} {r.Model} {r.Trim} {r.Type} {r.Cylinder} cylinders',axis=1).tolist()
col.add(
    ids=df.index.astype(str).tolist(),
    documents=docs,
    metadatas=df.to_dict('records')
)
print('Indexed',len(df),'cars')


### Soru


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
from IPython.display import Markdown
client=OpenAI(base_url='https://openrouter.ai/api/v1',api_key=os.getenv('OPENROUTER_API_KEY'))
def csv_bot(q):
    ctx='\n'.join(col.query(query_texts=[q],n_results=3)['documents'][0])
    r=client.chat.completions.create(model='openai/gpt-oss-120b:free',messages=[{'role':'user','content':f'CSV chatbot. ONLY context.\n{ctx}\n\nUser:{q}'}])
    return Markdown(r.choices[0].message.content)
csv_bot('4 cylinder sedan with high mileage')


### Sonuc

4 silindir sorusuna tablodan araba dondu.
